In [1]:
import pandas as pd 
import scanpy as sc 
import numpy as np
import snapatac2 as snap
from collections import Counter

In [2]:
def stratified_sample_by_sex_age(adata, celltype_key="final_cell_type", sex_key="sex", age_key="age_group", n_per_type=5000, random_state=0):
    '''Sample evenly n cells per cell type by sex and age group'''
    
    np.random.seed(random_state)
    
    obs = adata.obs.copy()
    sampled_indices = []

    for ct, df_ct in obs.groupby(celltype_key):
        # total cells needed for this cell type
        n_target = n_per_type
        
        # group within cell type by sex and age_group
        groups = df_ct.groupby([sex_key, age_key])
        n_groups = len(groups)
        
        # cells to take from each group evenly
        base_n = n_target // n_groups
        remainder = n_target % n_groups
        
        for i, (_, df_group) in enumerate(groups):
            n = base_n + (1 if i < remainder else 0)  # distribute remainder
            n = min(n, len(df_group))  # can't take more than available
            sampled_indices.extend(np.random.choice(df_group.index, size=n, replace=False))
    
    return adata[sampled_indices].copy()

In [3]:
%%time
adata = sc.read_h5ad("../07_final_ATAC.h5ad")
adata

CPU times: user 4.06 s, sys: 30 s, total: 34.1 s
Wall time: 44.5 s


AnnData object with n_obs × n_vars = 690044 × 654221
    obs: 'ATAC_barcode', 'sample_id', 'leiden', 'donor_id', 'study', 'age_status', 'age', 'sex', 'region', 'disease_binary', 'technology', 'fragment_file', 'full_path', 'file', 'nfrag', 'tsse', 'cell_type', 'tech_plus_study', 'age_group', 'decade', 'final_cell_type', 'cell_or_nuclei', 'disease'
    var: 'count', 'selected'
    uns: 'age_status_colors', 'cell_type_colors', 'leiden', 'leiden_colors', 'neighbors', 'spectral_eigenvalue', 'study_colors'
    obsm: 'X_spectral', 'X_spectral_harmony', 'X_umap'
    obsp: 'connectivities', 'distances'

### Filter to non-diseased postnatal donors, sampling evenly by sex and age status

In [4]:
filt_data = adata[(adata.obs.disease_binary == "N") & (adata.obs.age_status == "postnatal")].copy()

In [5]:
subsampled_adata = stratified_sample_by_sex_age(filt_data, n_per_type=3000)

In [6]:
# add sex status and age status
subsampled_adata.obs['sex_and_age_status'] = subsampled_adata.obs['sex'].astype(str) + ":" + subsampled_adata.obs['age_group'].astype(str)

In [7]:
adata_metadata = subsampled_adata.obs 
adata_metadata.groupby(["sex_and_age_status", "final_cell_type"])['ATAC_barcode'].count()

sex_and_age_status  final_cell_type
female:middle       Adipocyte            0
                    Cardiomyocyte      500
                    Endothelial        500
                    Epicardial         490
                    Fibroblast         500
                                      ... 
male:young          Mast               481
                    Myeloid            500
                    Neuronal           500
                    Pericyte           500
                    vSMC               500
Name: ATAC_barcode, Length: 66, dtype: int64

In [8]:
### filter to the cell types for this analysis
cell_types = ["Cardiomyocyte", "Endothelial", "Fibroblast", "Myeloid", "Pericyte"]

adata_subsampled = subsampled_adata[subsampled_adata.obs.final_cell_type.isin(cell_types)].copy()
adata_subsampled.shape

(15000, 654221)

In [12]:
adata_subsampled.shape

(15000, 654221)

In [14]:
Counter(adata_subsampled.obs.final_cell_type)

Counter({'Cardiomyocyte': 3000,
         'Endothelial': 3000,
         'Fibroblast': 3000,
         'Myeloid': 3000,
         'Pericyte': 3000})

In [11]:
adata_subsampled.write("01_subsampled_ATAC.h5ad")